This code aims to take a shapefile of station locations and measuements and removes a subset of stations at random to rerun in Greg's code. This could be implemented in some sort of potential cross validation.

In [1]:
import sys
import os as os

import geopandas as gpd
import numpy as np
from numpy.polynomial import Polynomial
import psycopg2
from netCDF4 import Dataset

from tqdm import tqdm
from multiprocessing import Pool
import statsmodels.api as sm
from scipy import stats
# from scipy import optimize

import cartopy.crs as ccrs
from cartopy.feature import NaturalEarthFeature as cfNEF

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import patches
#import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D

import rasterio
import xarray as xr

import random
import subprocess
import netCDF4
import shutil

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import ShuffleSplit

import pathlib
import glob
import pandas as pd

from sklearn.metrics import r2_score

The following code looks at all the shapefiles within the assimilations and has an ouput of the number of each type of station. This is to help decide which assimilations are worth potentially researching. It was saved in src folder.

In [10]:
# Read the CSV file
UMRB_ID = pd.read_csv(r'C:\Repos\umrb-snodas\resources\Current_UMRB_Station_List.csv')

# Create station ID List
UMRB_ID = UMRB_ID['station_id'].to_list()

In [12]:
# Specify the folder to look through
fldr = 'C:/Users/clemasters/Research Triangle Institute/CIROH Basecamp - Documents/Projects/0218723.014 - SNODAS/Data/snodas_assims_2023_2024'

# Create a list of file paths to shapefiles in the specified folder
files = [str(file) for file in pathlib.Path(fldr).rglob('*.shp')]

# Dictionary to store counts for each file
file_counts = {}

for file in files:
    # Read each shapefile
    gdf = gpd.read_file(file)
    
    # Check if 'STATION_ID' column exists in the GeoDataFrame
    if 'STATION_ID' in gdf.columns:
        # Count the occurrences of rows with station IDs present in the UMRB list
        count = gdf[gdf['STATION_ID'].isin(UMRB_ID)].shape[0]
        
        # Store the count in the dictionary with the filename as key
        file_counts[file] = count
    else:
        print(f"File {file} does not contain 'STATION_ID' column.")

# Convert the dictionary to a DataFrame
result_df = pd.DataFrame(file_counts.items(), columns=['File', 'Count'])

File C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\snodas_assims_2023_2024\NO_snodas_assim_20221018\ssm1054_md_based_2022101712_2022101812_us.shp does not contain 'STATION_ID' column.
File C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\snodas_assims_2023_2024\NO_snodas_assim_20221019\ssm1054_md_based_2022101812_2022101912_east.shp does not contain 'STATION_ID' column.


In [13]:
# Save the result DataFrame to a CSV file
result_df.to_csv('station_id_counts.csv', index=False)

The below code shows which files contain UMRB stations and the count of them in each shapefile as well as a snotel one to compare.

In [ ]:
UMRB = result_df.loc[result_df['STATION_TY'].str.contains('UMRB')] # Only 36 files contain UMRB stations with most around only 1 staion and the highest being 15 stations

In [ ]:
UMRB_CSV = pd.read_csv('C:/Repos/umrb-snodas/src/station_type_counts.csv')

SNOTEL_Number = UMRB_CSV.loc[UMRB_CSV['STATION_TY'].str.contains('SNOTEL')]



This code uses a shuffle split cross validator to remove test stations from shapefile and then runs an assimilation using IDW interpolation. The output files consist of a nudging layer image and a netCDF file showing the data from that nudging layer. They are saved within the src folder which can then be moved manually to a different folder of choice:

In [ ]:
# Inputs for code:
folder_name = "snodas_assim_20221214"
start_date = "2022121312" # These are the numbers within the shapefile like shown in shapefile name above
end_date = "2022121412" # These are the numbers within the shapefile like shown in shapefile name above

First run a baseline run using original shapefiles to get the new baseline netCDF file you will use for comparisons with other runs.

In [ ]:
## Run Baseline IDW Run ##

# Enter command you want to run (Look at commands in Greg's Examples, this command uses assigned IDW creteria to create new nudging layer) 
cmd = f"python ./snodas_idw.py -k -i 3 50 1 0 250 {start_date} {end_date} west -f ../Test_Runs/{folder_name}" #Careful with the US, West, etc. part in command

# Run Greg's code using the above inputs and chosen command in cmd line
subprocess.run(cmd, capture_output=True)

In [ ]:
# Inputs for code:
net_CDF1_Path = "C:/Repos/umrb-snodas/Test_Runs/snodas_assim_20221214_Baseline/ssm1054_snodas_idw_test_2022121412_west.nc" 
net_CDF2_Path = "C:/Users/clemasters/Research Triangle Institute/CIROH Basecamp - Documents/Projects/0218723.014 - SNODAS/Data/snodas_assims_2023_2024/snodas_assim_20221214/ssm_process_region_2022121312_2022121412_swe_west.nc"
net_CDF_name_us = "ssm_process_region_2022121312_2022121412_swe_west.nc"
net_CDF_name = "ssm1054_2022121412.nc" # When changing to baseline output netCDF you still need to save with the name of original example run name!!
shapefile_name = "ssm1054_md_based_2022121312_2022121412_west.shp"

# Load the shapefile:
shapefile_path = f"C:/Users/clemasters/Research Triangle Institute/CIROH Basecamp - Documents/Projects/0218723.014 - SNODAS/Data/snodas_assims_2023_2024/{folder_name}/{shapefile_name}"
gdf = gpd.read_file(shapefile_path)

# Filter out rows rows with 'SNOTEL' in 'STATION_TY' switch this with UMRB Stations!!!!
gdf_filtered = gdf[gdf['STATION_TY'] == 'MESO-UMRB']

# Initialize ShuffleSplit
sss = ShuffleSplit(n_splits=1, test_size=0.10, random_state=0) # n_splits define how many test runs to do and test_size takes a percentage of gdf_filtered as test group

# Generate indices for splitting
for i, (train_index, test_index) in enumerate(sss.split(gdf_filtered)):  # Use gdf_filtered for generating test indices
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")
    
    # Get latitude and longitude values from test subset in gdf_filtered
    test_latitudes = gdf_filtered.iloc[test_index]['LATITUDE']
    test_longitudes = gdf_filtered.iloc[test_index]['LONGITUDE']
    
    # Remove rows from gdf using latitude and longitude values
    train_subset = gdf[~((gdf['LATITUDE'].isin(test_latitudes)) & (gdf['LONGITUDE'].isin(test_longitudes)))]
    
    # Extract test subset from gdf_filtered
    test_subset = gdf_filtered.iloc[test_index]
    
    # Print the size of each subset
    print(f"  Train subset size: {len(train_subset)}")
    print(f"  Test subset size: {len(test_subset)}")
    
    # Change folder and shapefile names
    folder_name = f"{folder_name}{i}"
    shapefile_name = f"ssm1054_md_based_2022121312_2022121412_west{i}.shp" # Change shapefile name
    net_CDF_name_us = f"ssm_process_region_2022121312_2022121412_swe_west{i}.nc" # Change processing region netCDF name
    
    
    # Write new shapefile to new folder - had to save twice to get correct shapefile name - Ideas???
    train_subset.to_file(f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}")
    train_subset.to_file(f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}/{shapefile_name}")
    
        
    # Copy netCDF's from orgignal folder to new folder (Need original netCDF's to run)
    shutil.copy(net_CDF1_Path,f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}/{net_CDF_name}" )
    shutil.copy(net_CDF2_Path,f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}/{net_CDF_name_us}" )
    
    # Enter command you want to run (Look at commands in Greg's Examples, this command uses assigned IDW creteria to create new nudging layer) 
    cmd = f"python ./snodas_idw.py -k -i 3 50 1 5 250 {start_date} {end_date} west{i} -f ../Test_Runs/{folder_name}" #Careful with the US, West, etc. part in command
    
    # Run Greg's code using the above inputs and chosen command in cmd line
    subprocess.run(cmd, capture_output=True)

The below code is designed for cross validation on the test assimilations. It compares the IDW values to the observed station values and finds how well the modeled IDW values fit the observed station values using the objective function R^2.

In [ ]:
# Output netCDF file with removed stations
data_netCDF_Output = xr.open_dataset('C:/Repos/umrb-snodas/Test_Runs/snodas_assim_202212140/ssm1054_snodas_idw_test_2022121412_west0.nc')

# Extract latitude, longitude, and data variables
latitude = data_netCDF_Output['lat']
longitude = data_netCDF_Output['lon']
data = data_netCDF_Output['Data']  # Replace 'Data' with the actual name of your data variable

# Convert data to NumPy arrays
data_array = data.values

# Function to find the nearest grid point in the netCDF array
def find_nearest_index(lat, lon, lat_values, lon_values):
    lat_index = np.abs(lat_values - lat).argmin()
    lon_index = np.abs(lon_values - lon).argmin()
    return lat_index, lon_index

# Lists to store observed and NetCDF values
observed_values = []
netcdf_values = []

# Iterate through each point in the test subset
for index, row in test_subset.iterrows():
    lat = row['LATITUDE']  # Assuming 'LATITUDE' is the column name in the GeoDataFrame
    lon = row['LONGITUDE']  # Assuming 'LONGITUDE' is the column name in the GeoDataFrame
    
    # Find the nearest grid point in the netCDF array
    lat_index, lon_index = find_nearest_index(lat, lon, latitude, longitude)
    
    # Extract the value from the netCDF array
    value = data_array[lat_index, lon_index]
    
    # Extract the observed value from the test subset
    observed_value = row['D_SWE_OM']  # Replace 'observed_value_column' with the actual column name
    
    # Append values to lists
    observed_values.append(observed_value)
    netcdf_values.append(value)

# Calculate R^2 value
r_squared = r2_score(observed_values, netcdf_values)

print("R^2 value:", r_squared)


